In [1]:
import numpy as np, pandas as pd, scanpy as sc, matplotlib.pyplot as plt, os
from scipy.stats import hypergeom
import celloracle as co, glob, pickle
from functools import reduce
from tqdm import tqdm
import itertools, math, random
import networkx as nx

# visualization settings required to see plots in jupyter notebook
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
plt.rcParams['figure.figsize'] = [6, 4.5]
plt.rcParams["savefig.dpi"] = 300

/ocean/projects/cis240075p/asachan/.conda/envs/celloracle_env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [ ]:
# from state_lf_enrich import StateSpecificEnrichment
# from utils import *

In [ ]:
import ast
import glob
import itertools
import math
import os
import pickle
import random
from typing import Optional

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scanpy as sc
from scipy.stats import hypergeom
from tqdm import tqdm

import celloracle as co

from utils import (
    create_combined_links_for_cluster_fusion,
    filter_combined_links_and_build_grn,
    filter_network_score_data,
    enrichment,
    get_SLIDE_GRN_enrichment,
    create_enrichment_df,
    _create_enrich_df,
)


class StateSpecificEnrichment:
    """
      1. load GRN + SLIDE data
      2. build combined links per cluster fusion
      3. construct TF lists (SLIDE-informed, network-matched, random)
      4. run hypergeometric enrichment
      5. post-process and merge weights
      6. visualise results (bar charts, TF-centric networks)
    """

    def __init__(
        self,
        grn_wd: str,
        oracle_object_name: str,
        feature_folder: str,
        out_path: str,
        experiment: str,
        slide_starting_genes: int,
        clusters_of_interest: list[str],
        order_fr_clust: list[int],
        order_fr_tfcomb: list[int],
        weight: str = "strength",
        quantile: float = 0.70,
        seed: int = 42,
    ):
        self.grn_wd = grn_wd
        self.oracle_object_name = oracle_object_name
        self.feature_folder = feature_folder
        self.out_path = out_path
        self.experiment = experiment
        self.slide_starting_genes = slide_starting_genes
        self.clusters_of_interest = clusters_of_interest
        self.order_fr_clust = order_fr_clust
        self.order_fr_tfcomb = order_fr_tfcomb
        self.weight = weight
        self.quantile = quantile

        random.seed(seed)
        os.makedirs(f"{out_path}/figures", exist_ok=True)
        os.makedirs(f"{out_path}/out_files", exist_ok=True)
        os.makedirs(f"{out_path}/out_files/SLIDE_LF_enrichment", exist_ok=True)
        sc.settings.figdir = f"{out_path}/figures"
        plt.rcParams["figure.figsize"] = [6, 4.5]
        plt.rcParams["savefig.dpi"] = 300

        self.GRN_links_after_fit = None
        self.GRN_network_scores = None
        self.GRN_TFs = None
        self.slide_features = None
        self.cluster_fusions = []
        self.cc_dicts: dict[tuple, dict[int, dict]] = {}
        self.enrichment_dfs: dict[tuple, dict[int, pd.DataFrame]] = {}

    # ── 1. Data loading ────────────────────────────────────────────────────

    def load_grn_data(self):
        oracle = co.load_hdf5(
            f"{self.grn_wd}/out_files/{self.oracle_object_name}"
        )
        self.GRN_network_scores = pd.read_csv(
            f"{self.grn_wd}/out_files/ridge_fitted_2_merged_network_scores.csv",
            index_col=0,
        )
        self.GRN_TFs = oracle.all_regulatory_genes_in_TFdict
        self.GRN_links_after_fit = {
            key: [] for key in oracle.coef_matrix_per_cluster.keys()
        }
        for cluster in oracle.coef_matrix_per_cluster.keys():
            links = (
                oracle.coef_matrix_per_cluster[cluster]
                .stack()
                .reset_index()
            )
            links.columns = ["source", "target", "coef_mean"]
            links = links[links["coef_mean"] != 0].reset_index(drop=True)
            links["coef_abs"] = np.abs(links["coef_mean"])
            self.GRN_links_after_fit[cluster] = links

    def load_slide_features(self):
        feature_files = glob.glob(f"{self.feature_folder}/*feature_list*")
        if self.experiment == "male_type2":
            data = [
                pd.read_csv(f, sep="\t", header=0)
                for f in feature_files
                if "Z15" in f or "Z29" in f
            ]
        elif self.experiment == "female_type2":
            data = [
                pd.read_csv(f, sep="\t", header=0)
                for f in feature_files
                if "Z15" in f or "Z12" in f
            ]
        else:
            raise ValueError(f"Unknown experiment: {self.experiment}")
        self.slide_features = set(pd.concat(data)["names"])

    def load_slide_features_with_corr(self):
        feature_files = glob.glob(f"{self.feature_folder}/*feature_list*")
        if self.experiment == "male_type2":
            data = [
                pd.read_csv(f, sep="\t", header=0)
                for f in feature_files
                if "Z15" in f or "Z29" in f
            ]
        elif self.experiment == "female_type2":
            data = [
                pd.read_csv(f, sep="\t", header=0)
                for f in feature_files
                if "Z15" in f or "Z38" in f or "Z42" in f or "Z55" in f
            ]
        else:
            raise ValueError(f"Unknown experiment: {self.experiment}")
        df = pd.concat(data)
        self.positive_corr_names = set(df[df["corrs"] >= 0]["names"])
        self.negative_corr_names = set(df[df["corrs"] < 0]["names"])

    def load_slide_features_from_files(
        self,
        feature_files: list[str] | None = None,
        a_loading_thresholds: list[float] | float | None = None,
    ):
        if feature_files is None:
            feature_files = glob.glob(f"{self.feature_folder}/*feature_list*")

        if isinstance(a_loading_thresholds, (int, float)):
            a_loading_thresholds = [a_loading_thresholds] * len(feature_files)

        if a_loading_thresholds is not None and len(a_loading_thresholds) != len(feature_files):
            raise ValueError(
                f"Length mismatch: {len(feature_files)} files but "
                f"{len(a_loading_thresholds)} thresholds."
            )

        frames = []
        for i, f in enumerate(feature_files):
            df = pd.read_csv(f, sep="\t", header=0)
            if a_loading_thresholds is not None:
                df = df[df["A_loading"] >= a_loading_thresholds[i]]
            frames.append(df)

        self.slide_features = set(pd.concat(frames)["names"])

    # ── 2. Run enrichment ──────────────────────────────────────────────────

    def build_cluster_fusions(self):
        self.cluster_fusions = []
        for ord_clus in self.order_fr_clust:
            self.cluster_fusions += list(
                itertools.combinations(self.clusters_of_interest, ord_clus)
            )

    def run_enrichment(self, save_pickle: bool = True):
        self.build_cluster_fusions()
        for cluster_fusion in self.cluster_fusions:
            combined_links, threshold = create_combined_links_for_cluster_fusion(
                self.out_path,
                cluster_fusion,
                self.GRN_links_after_fit,
                quantile=self.quantile,
            )
            grn, edges_df = filter_combined_links_and_build_grn(
                combined_links, threshold
            )

            fig = (
                edges_df.groupby(["strength", "key"])
                .size()
                .unstack()
                .plot(kind="bar", stacked=True)
            )
            plt.xlabel("Strength")
            plt.ylabel("Count")
            plt.title("Distribution of Key and Strength")
            plt.savefig(
                f"{self.out_path}/figures/combined_links_key_strength_{cluster_fusion}{self.experiment}.pdf"
            )
            plt.close()

            slide_features = self.slide_features.intersection(set(grn.nodes))
            slide_features_neighbors = []
            for gene in slide_features:
                slide_features_neighbors += list(grn.predecessors(gene))
            slide_tot_TF = (
                slide_features.union(set(slide_features_neighbors))
            ).intersection(self.GRN_TFs)

            for ord_tf in self.order_fr_tfcomb:
                cc_dict = {}
                if ord_tf == 1:
                    cases = [(slide_tot_TF, "slide")]
                elif ord_tf == 2:
                    cases = [(slide_tot_TF, "slide")]
                else:
                    raise ValueError(f"Unsupported TF combination order: {ord_tf}")

                for TFs, case in cases:
                    print(
                        f"Running enrichment for {self.experiment}, "
                        f"{cluster_fusion}, {ord_tf} TFs, case = {case}"
                    )
                    cc_dict = get_SLIDE_GRN_enrichment(
                        edges_df,
                        cc_dict,
                        cluster_fusion,
                        ord_tf,
                        slide_features,
                        self.slide_starting_genes,
                        TFs,
                        case,
                    )

                self.cc_dicts.setdefault(cluster_fusion, {})[ord_tf] = cc_dict

                if save_pickle:
                    suffix = f"_{self.experiment}"
                    print(
                        f"Creating enrichment dataframe for {self.experiment}, "
                        f"{cluster_fusion}, {ord_tf} TFs, all cases"
                    )
                    enrichment_df = create_enrichment_df(
                        self.out_path,
                        cc_dict,
                        cluster_fusion,
                        ord_tf,
                        filter=None,
                        suffix=suffix,
                    )
                    self.enrichment_dfs.setdefault(cluster_fusion, {})[
                        ord_tf
                    ] = enrichment_df
                    print(enrichment_df["case"].value_counts())
                    pkl_path = (
                        f"{self.out_path}/out_files/SLIDE_LF_enrichment/"
                        f"cc_dict_{ord_tf}_TFs_{cluster_fusion}{suffix}.pickle"
                    )
                    with open(pkl_path, "wb") as f:
                        pickle.dump(cc_dict, f)

    # ── 3. Post-processing ─────────────────────────────────────────────────

    def load_results(self, cluster_fusion: tuple, ord_tf: int):
        suffix = f"_{self.experiment}"
        pkl_path = (
            f"{self.out_path}/out_files/SLIDE_LF_enrichment/"
            f"cc_dict_{ord_tf}_TFs_{cluster_fusion}{suffix}.pickle"
        )
        with open(pkl_path, "rb") as f:
            cc_dict = pickle.load(f)
        self.cc_dicts.setdefault(cluster_fusion, {})[ord_tf] = cc_dict

        csv_path = (
            f"{self.out_path}/out_files/SLIDE_LF_enrichment/"
            f"enriched_df_{ord_tf}_TFs_{cluster_fusion}{suffix}.csv"
        )
        self.enrichment_dfs.setdefault(cluster_fusion, {})[ord_tf] = pd.read_csv(
            csv_path
        )

    def build_used_weight_df(self, cluster_fusion: tuple, ord_tf: int) -> pd.DataFrame:
        suffix = f"_{self.experiment}"
        cc_dict = self.cc_dicts[cluster_fusion][ord_tf]
        concatenated = pd.DataFrame()
        for i in range(len(cc_dict[cluster_fusion][ord_tf])):
            w_df = cc_dict[cluster_fusion][ord_tf][i][5][1]
            if w_df.empty:
                continue
            w_df["case"] = cc_dict[cluster_fusion][ord_tf][i][4]
            concatenated = pd.concat([concatenated, w_df], ignore_index=True)
        out = (
            f"{self.out_path}/out_files/SLIDE_LF_enrichment/"
            f"{ord_tf}_TFs_{cluster_fusion}{suffix}_used_weights.csv"
        )
        concatenated.to_csv(out, index=False)
        return concatenated

    def build_slide_lf_enriched(
        self, cluster_fusion: tuple, ord_tf: int
    ) -> pd.DataFrame:
        suffix = f"_{self.experiment}"
        enr_path = (
            f"{self.out_path}/out_files/SLIDE_LF_enrichment/"
            f"enriched_df_{ord_tf}_TFs_{cluster_fusion}{suffix}.csv"
        )
        wt_path = (
            f"{self.out_path}/out_files/SLIDE_LF_enrichment/"
            f"{ord_tf}_TFs_{cluster_fusion}{suffix}_used_weights.csv"
        )
        slide_lf = pd.read_csv(enr_path)
        used_w = pd.read_csv(wt_path)
        slide_lf = slide_lf[slide_lf["case"] == "slide"]
        used_w = used_w[used_w["case"] == "slide"]

        slide_lf = slide_lf[["TF", "common"]]
        slide_lf["TF"] = slide_lf["TF"].apply(ast.literal_eval).apply(tuple)
        slide_lf["common"] = slide_lf["common"].apply(ast.literal_eval).apply(list)
        slide_lf = slide_lf.explode("common").explode("TF")
        slide_lf.columns = ["source", "target"]
        used_w = used_w[used_w["strength"] == 1]
        slide_lf = slide_lf.merge(used_w, on=["source", "target"], how="left")

        out = (
            f"{self.out_path}/out_files/SLIDE_LF_enrichment/"
            f"enriched_df_{ord_tf}_TFs_{cluster_fusion}{suffix}_used_weights_strong.csv"
        )
        slide_lf.to_csv(out, index=False)
        return slide_lf

    # ── 4. Visualization ───────────────────────────────────────────────────

    def plot_strength_key_distribution(
        self, slide_lf_enriched: pd.DataFrame, cluster_fusion: tuple, ord_tf: int
    ):
        suffix = f"_{self.experiment}"
        slide_lf_enriched.groupby(["strength", "key"]).size().unstack().plot(
            kind="bar", stacked=True
        )
        plt.xlabel("Strength")
        plt.ylabel("Count")
        plt.title("Distribution of Key and Strength")
        plt.savefig(
            f"{self.out_path}/figures/"
            f"enriched_key_strength_{ord_tf}_TFs_{cluster_fusion}{suffix}_used_weights_strong.pdf"
        )
        plt.close()

    def plot_enrichment_scores_with_color_proportions(
        self,
        slide_lf_enriched: pd.DataFrame,
        cluster_fusion: tuple,
        ord_tf: int,
    ):
        if not hasattr(self, "positive_corr_names"):
            self.load_slide_features_with_corr()

        suffix = f"_{self.experiment}"
        cc_dict = self.cc_dicts[cluster_fusion][ord_tf]

        lf = slide_lf_enriched.copy()
        lf["color"] = lf["target"].apply(
            lambda g: "red"
            if g in self.positive_corr_names
            else ("blue" if g in self.negative_corr_names else "gray")
        )
        color_hist = (
            lf.groupby("source")["color"]
            .value_counts(normalize=True)
            .unstack(fill_value=0)
        )

        tf_name_score = {k: None for k in color_hist.index.values}
        for entry in cc_dict[cluster_fusion][ord_tf]:
            tf, cond, score_val = entry[0][0], entry[1], entry[2][0]
            if tf in tf_name_score and cond == (1,):
                tf_name_score[tf] = score_val

        sorted_data = sorted(
            [(v, k) for k, v in tf_name_score.items() if v is not None],
            reverse=True,
            key=lambda x: x[0],
        )
        if not sorted_data:
            raise ValueError("No valid TF scores found for plotting.")

        score, tf_name = zip(*sorted_data)
        score_series = pd.Series(score, index=tf_name, name="score")
        color_proportions = color_hist.loc[list(tf_name)]
        original_props = color_proportions.copy()
        scaled = color_proportions.mul(score_series, axis=0).reset_index()
        scaled = scaled.rename(columns={"index": "source"})

        melted_scaled = scaled.melt(id_vars="source", var_name="color", value_name="height")
        melted_props = (
            original_props.reset_index().melt(
                id_vars="source", var_name="color", value_name="proportion"
            )
        )
        plot_df = melted_scaled.merge(melted_props, on=["source", "color"])
        plot_df["text"] = (plot_df["proportion"] * 100).round(1).astype(str) + "%"

        fig = px.bar(
            plot_df,
            x="source",
            y="height",
            color="color",
            text="text",
            labels={"height": "Score-scaled Proportion", "source": "TF"},
            title=(
                f"TF Enrichment Scores with color Proportions for "
                f"{self.experiment} - {cluster_fusion} - {ord_tf} Order"
            ),
        )
        fig.update_traces(textposition="inside", insidetextanchor="middle")
        fig.update_layout(
            xaxis_title="Transcription Factor (TF)",
            yaxis_title="Enrichment Score",
            xaxis_tickangle=-90,
            font=dict(family="Arial", size=8, color="black"),
            plot_bgcolor="rgba(0,0,0,0)",
            paper_bgcolor="white",
            xaxis=dict(showgrid=False, showline=True, linecolor="black", ticks="outside"),
            yaxis=dict(showgrid=False, showline=True, linecolor="black", ticks="outside"),
        )
        fig.write_image(
            f"{self.out_path}/figures/"
            f"0_7_TF_Enrichment_Scores_with_color_Proportions_{ord_tf}_TFs_{cluster_fusion}_{self.experiment}.svg",
            format="svg",
        )
        return fig


In [3]:
wd = '/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/grn_CO/female_type2'
feature_folder = '/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/slide_outs/optimize_slide/female_0.1_spec/0.01_0.1_out'
out_path = os.path.join(wd, 'lf_enrich_small')
os.makedirs(f"{out_path}/figures", exist_ok=True)
os.makedirs(f"{out_path}/out_files", exist_ok=True)
os.makedirs(f"{out_path}/out_files/SLIDE_LF_enrichment", exist_ok=True)
sc.settings.figdir = f"{out_path}/figures"
random.seed(42)

## Filter the LF genes based on loadings

In [4]:
sse = StateSpecificEnrichment(
    grn_wd=wd,
    oracle_object_name="female_type2_fitted.celloracle.oracle",
    feature_folder=feature_folder,
    out_path=os.path.join(wd, "lf_enrich_small"),
    experiment="female_type2",
    slide_starting_genes=1323,
    clusters_of_interest=["KO", "WT"],
    order_fr_clust=[2],
    order_fr_tfcomb=[1],
    weight="strength",
    quantile=0.70,
)

In [5]:
sse.load_grn_data

<bound method StateSpecificEnrichment.load_grn_data of <__main__.StateSpecificEnrichment object at 0x1519231fefe0>>

In [ ]:
sse.load_slide_features_from_files(feature_files=['/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/slide_outs/optimize_slide/female_0.1_spec/0.01_0.1_out/feature_list_Z12.txt', '/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/slide_outs/optimize_slide/female_0.1_spec/0.01_0.1_out/feature_list_Z15.txt'], a_loading_threshold=0.01)

In [ ]:
# 1. Load data
sse.load_data()

# 2. Run enrichment
sse.run_enrichment(save_pickle=True)

# 3. Post-process (or load from disk)
cluster_fusion = ("KO", "WT")
ord_tf = 1
# sse.load_results(cluster_fusion, ord_tf)  # alternative: load from saved files
sse.build_used_weight_df(cluster_fusion, ord_tf)
slide_lf = sse.build_slide_lf_enriched(cluster_fusion, ord_tf)

# 4. Plots
sse.plot_strength_key_distribution(slide_lf, cluster_fusion, ord_tf)
sse.plot_enrichment_scores_with_color_proportions(slide_lf, cluster_fusion, ord_tf)

In [ ]:
# ── 5. TF-centric network visualization ────────────────────────────────
@staticmethod
def build_tf_edge_df(
    enrichment_df: pd.DataFrame,
    used_weight_df: pd.DataFrame,
    tf: str,
    condition: str = "KO",
    strength: Optional[int] = 1,
    gene_set: str = "all",
    fallback: bool = True,
) -> pd.DataFrame:
    gene_set = gene_set.lower()
    col = "common" if gene_set == "common" else "dwnstrm"

    enr = enrichment_df.copy()
    if enr["TF"].dtype == object:
        enr["TF"] = enr["TF"].apply(
            lambda v: ast.literal_eval(v) if isinstance(v, str) else v
        )
        enr["TF"] = enr["TF"].apply(
            lambda v: v[0] if isinstance(v, (tuple, list)) and len(v) == 1 else v
        )

    row = enr[enr["TF"].str.lower() == tf.lower()]
    if row.empty:
        raise ValueError(f"TF '{tf}' not found in enrichment_df.")

    raw = row.iloc[0][col]
    if isinstance(raw, str):
        raw = ast.literal_eval(raw)
    gene_list = list(raw)

    w = used_weight_df.copy()
    w = w[w["source"].str.lower() == tf.lower()]
    w = w[w["target"].isin(gene_list)]
    if strength is not None:
        w = w[w["strength"] == strength]

    if condition == "both":
        w = (
            w.assign(_abs=w["weight"].abs())
            .sort_values("_abs", ascending=False)
            .drop_duplicates(subset=["source", "target", "cluster"])
            .drop(columns="_abs")
            .reset_index(drop=True)
        )
        w["used_condition"] = w["cluster"]
        return w

    other = "WT" if condition == "KO" else "KO"
    records = []
    for gene in gene_list:
        g = w[w["target"] == gene]
        primary = g[g["cluster"] == condition]
        if not primary.empty:
            best = primary.loc[primary["weight"].abs().idxmax()].to_dict()
            best["used_condition"] = condition
            records.append(best)
        elif fallback:
            secondary = g[g["cluster"] == other]
            if not secondary.empty:
                best = secondary.loc[secondary["weight"].abs().idxmax()].to_dict()
                best["used_condition"] = other
                records.append(best)

    if not records:
        raise ValueError(
            f"No edges resolved for TF='{tf}', condition='{condition}'."
        )
    return pd.DataFrame(records).reset_index(drop=True)

@staticmethod
def plot_tf_network(
    enrichment_df: pd.DataFrame,
    used_weight_df: pd.DataFrame,
    tf: str,
    condition: str = "KO",
    strength: Optional[int] = 1,
    gene_set: str = "all",
    highlight_common: bool = True,
    pos_color: str = "#E24B4A",
    neg_color: str = "#378ADD",
    tf_node_color: str = "#7F77DD",
    common_border: str = "#FAC775",
    min_width: float = 1.5,
    max_width: float = 14.0,
    node_size: int = 20,
    height: int = 750,
    title: Optional[str] = None,
    save_path: Optional[str] = None,
    show: bool = False,
) -> go.Figure:
    edges = StateSpecificEnrichment.build_tf_edge_df(
        enrichment_df, used_weight_df, tf, condition, strength, gene_set
    )

    common_genes: set = set()
    if highlight_common and gene_set == "all":
        enr = enrichment_df.copy()
        if enr["TF"].dtype == object:
            enr["TF"] = enr["TF"].apply(
                lambda v: ast.literal_eval(v) if isinstance(v, str) else v
            )
            enr["TF"] = enr["TF"].apply(
                lambda v: v[0]
                if isinstance(v, (tuple, list)) and len(v) == 1
                else v
            )
        r = enr[enr["TF"].str.lower() == tf.lower()]
        if not r.empty:
            raw = r.iloc[0]["common"]
            common_genes = set(
                ast.literal_eval(raw) if isinstance(raw, str) else raw
            )

    all_targets = edges["target"].unique().tolist()
    n = len(all_targets)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
    pos = {tf: np.array([0.0, 0.0])}
    for gene, angle in zip(all_targets, angles):
        pos[gene] = np.array([np.cos(angle), np.sin(angle)])

    w_abs = edges["weight"].abs()
    w_min, w_max = w_abs.min(), w_abs.max()
    span = w_max - w_min if w_max > w_min else 1.0

    edge_traces, annotations = [], []
    for _, r in edges.iterrows():
        src, tgt, w = r["source"], r["target"], r["weight"]
        used = r["used_condition"]
        x0, y0 = pos[src]
        x1, y1 = pos[tgt]
        norm = (abs(w) - w_min) / span
        width = min_width + norm * (max_width - min_width)
        color = pos_color if w >= 0 else neg_color
        is_ko = used == "KO"
        dash = "solid" if is_ko else "dot"
        lw = width if is_ko else max(width * 0.65, min_width)

        edge_traces.append(
            go.Scatter(
                x=[x0, x1, None],
                y=[y0, y1, None],
                mode="lines",
                line=dict(width=lw, color=color, dash=dash),
                hovertemplate=(
                    f"<b>{src} -> {tgt}</b><br>weight: {w:.4f}<br>"
                    f"{'activation' if w >= 0 else 'repression'}<br>"
                    f"condition: {used}"
                    + (
                        " <i>(fallback)</i>"
                        if used != condition and condition != "both"
                        else ""
                    )
                    + "<extra></extra>"
                ),
                showlegend=False,
            )
        )
        frac = 0.78
        annotations.append(
            dict(
                x=x0 + frac * (x1 - x0),
                y=y0 + frac * (y1 - y0),
                ax=x0 + (frac - 0.12) * (x1 - x0),
                ay=y0 + (frac - 0.12) * (y1 - y0),
                xref="x",
                yref="y",
                axref="x",
                ayref="y",
                showarrow=True,
                arrowhead=2,
                arrowsize=1.2,
                arrowwidth=1.5,
                arrowcolor=color,
            )
        )

    node_w = (
        edges.assign(_abs=edges["weight"].abs())
        .sort_values("_abs", ascending=False)
        .drop_duplicates(subset=["target"])
        .set_index("target")["weight"]
    )

    tx, ty, tlabels, tcolors = [], [], [], []
    tborder_c, tborder_w, thovers = [], [], []
    for gene in all_targets:
        w = node_w[gene]
        is_common = gene in common_genes
        fill = pos_color if w >= 0 else neg_color
        bcolor = common_border if (highlight_common and is_common) else "white"
        bwidth = 3.0 if (highlight_common and is_common) else 1.5
        tx.append(pos[gene][0])
        ty.append(pos[gene][1])
        tlabels.append(f"{'* ' if is_common else ''}{gene}")
        tcolors.append(fill)
        tborder_c.append(bcolor)
        tborder_w.append(bwidth)
        thovers.append(
            f"<b>{gene}</b><br>weight: {w:.4f}<br>"
            f"{'activation' if w >= 0 else 'repression'}<br>"
            f"{'<i>common / LF-enriched</i>' if is_common else '<i>downstream only</i>'}"
            "<extra></extra>"
        )

    target_trace = go.Scatter(
        x=tx,
        y=ty,
        mode="markers+text",
        marker=dict(
            size=node_size,
            color=tcolors,
            line=dict(width=tborder_w, color=tborder_c),
        ),
        text=tlabels,
        textposition="top center",
        textfont=dict(size=10, family="Arial"),
        hovertemplate=thovers,
        showlegend=False,
    )
    tf_trace = go.Scatter(
        x=[0],
        y=[0],
        mode="markers+text",
        marker=dict(
            size=node_size * 2,
            color=tf_node_color,
            line=dict(width=2, color="white"),
        ),
        text=[tf],
        textposition="middle center",
        textfont=dict(size=13, color="white", family="Arial"),
        hovertemplate=f"<b>{tf}</b> (TF hub)<extra></extra>",
        showlegend=False,
    )

    legend = [
        go.Scatter(
            x=[None], y=[None], mode="lines",
            line=dict(width=4, color=pos_color), name="Activation (w > 0)",
        ),
        go.Scatter(
            x=[None], y=[None], mode="lines",
            line=dict(width=4, color=neg_color), name="Repression (w < 0)",
        ),
        go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(size=10, color=tf_node_color), name=f"{tf} (TF)",
        ),
    ]
    if condition == "both":
        legend += [
            go.Scatter(
                x=[None], y=[None], mode="lines",
                line=dict(width=3, color="gray", dash="solid"), name="KO (solid)",
            ),
            go.Scatter(
                x=[None], y=[None], mode="lines",
                line=dict(width=2, color="gray", dash="dot"), name="WT (dotted)",
            ),
        ]
    elif edges["used_condition"].ne(condition).any():
        other = "WT" if condition == "KO" else "KO"
        legend.append(
            go.Scatter(
                x=[None], y=[None], mode="lines",
                line=dict(width=2, color="gray", dash="dot"),
                name=f"fallback weight ({other})",
            )
        )
    if highlight_common and gene_set == "all":
        legend += [
            go.Scatter(
                x=[None], y=[None], mode="markers",
                marker=dict(
                    size=10, color="gray",
                    line=dict(width=3, color=common_border),
                ),
                name="* common / LF-enriched",
            ),
            go.Scatter(
                x=[None], y=[None], mode="markers",
                marker=dict(
                    size=10, color="gray", line=dict(width=1.5, color="white")
                ),
                name="downstream only",
            ),
        ]

    n_common = sum(g in common_genes for g in all_targets)
    auto_title = (
        f"<b>{tf}</b> -- {condition} | {gene_set} genes "
        f"({len(all_targets)} targets"
        + (f", {n_common} common" if gene_set == "all" else "")
        + ")"
    )

    fig = go.Figure(
        data=edge_traces + [target_trace, tf_trace] + legend,
        layout=go.Layout(
            title=dict(
                text=title or auto_title,
                font=dict(size=14, family="Arial"),
                x=0.5,
            ),
            height=height,
            showlegend=True,
            legend=dict(
                x=1.01,
                y=0.95,
                bgcolor="rgba(255,255,255,0.85)",
                bordercolor="lightgray",
                borderwidth=1,
                font=dict(size=11),
            ),
            hovermode="closest",
            xaxis=dict(
                showgrid=False,
                zeroline=False,
                showticklabels=False,
                scaleanchor="y",
            ),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            plot_bgcolor="white",
            paper_bgcolor="white",
            margin=dict(l=20, r=180, t=60, b=20),
            annotations=annotations,
        ),
    )
    if show:
        fig.show()
    if save_path:
        fig.write_image(save_path, format=save_path.rsplit(".", 1)[-1])
        print(f"Saved -> {save_path}")
    return fig


In [ ]:
# 5. TF-centric network
enrichment_df = sse.enrichment_dfs[cluster_fusion][ord_tf]
used_weight_df = pd.read_csv(
    f"{sse.out_path}/out_files/SLIDE_LF_enrichment/"
    f"{ord_tf}_TFs_{cluster_fusion}_{sse.experiment}_used_weights.csv"
)
fig = sse.plot_tf_network(
    enrichment_df, used_weight_df, tf="Foxa3", condition="KO",
    gene_set="all", highlight_common=True,
)

In [ ]:
def fetch_GRN_data(oracle_object_name, GRN_wd):
    # Read oracle links after fitting
    oracle = co.load_hdf5(f"{GRN_wd}/out_files/{oracle_object_name}")
    GRN_network_scores = pd.read_csv(f"{GRN_wd}/out_files/ridge_fitted_2_merged_network_scores.csv", index_col=0)
    GRN_TFs = oracle.all_regulatory_genes_in_TFdict
    GRN_links_after_fit = {key: [] for key in oracle.coef_matrix_per_cluster.keys()}
    for cluster in oracle.coef_matrix_per_cluster.keys():
        cluster_specific_links = oracle.coef_matrix_per_cluster[cluster].stack().reset_index()
        cluster_specific_links.columns = ['source', 'target', 'coef_mean']
        cluster_specific_links = cluster_specific_links[cluster_specific_links ['coef_mean'] != 0].reset_index(drop=True)
        cluster_specific_links['coef_abs'] = np.abs(cluster_specific_links['coef_mean'])
        GRN_links_after_fit[cluster] = cluster_specific_links
        # links_after_fit[cluster]['weight'] = links_after_fit[cluster]['coef_abs'] * links_after_fit[cluster]['-logp']
        # GRN_TFs = GRN_TFs + list(links_after_fit[cluster].source.unique())
    return GRN_links_after_fit, GRN_network_scores, GRN_TFs

In [ ]:
def read_slide_data(feature_folder, experiment):
    # Read SLIDE data for the experiment
    feature_files = glob.glob(f"{feature_folder}/*feature_list*")
    if experiment == 'male_type2':
        feature_data = [pd.read_csv(file, sep='\t', header=0) for file in feature_files if 'Z15' in file or 'Z29' in file]
    elif experiment == 'female_type2':
        #feature_data = [pd.read_csv(file, sep='\t', header = 0) for file in feature_files if 'Z15' in file or 'Z38' in file or 'Z42' in file or 'Z55' in file]
        feature_data = [pd.read_csv(file, sep='\t', header = 0) for file in feature_files if 'Z15' in file or 'Z12']
    feature_data = pd.concat(feature_data)
    slide_features = set(feature_data['names'])
    return slide_features

In [ ]:
# settings for enrichment analysis
input_dict = {
    'experiment': ['male_type2', 'female_type2'], # list of experiment names
    'slide_starting_genes': [1296, 1323], # number of starting genes in SLIDE (males and females)
    'clusters_of_interest': [['KO','WT'], ['KO','WT']], # list of lists of clusters
    'order_fr_clust': [[2], [2]], # list of lists of cluster combination orders: 2 means combinations of 2 clusters
    'order_fr_tfcomb': [[1], [1]], # list of lists of TF combination orders: 1 means single TF
    'weight': ['strength', 'strength'],
}
input_df = pd.DataFrame(input_dict)

In [ ]:
#### Assign the input parameters
#### You can change the index to 0 or 1 to run for different experiments
#### Or just loop over the dataframe rows
i=1
experiment = input_df['experiment'][i]
slide_starting_genes = input_df['slide_starting_genes'][i]
clusters_of_interest = input_df['clusters_of_interest'][i]
order_fr_clust = input_df['order_fr_clust'][i]
order_fr_tfcomb = input_df['order_fr_tfcomb'][i]
weight = input_df['weight'][i]

In [ ]:
# ------------------------------------------------------------
# Reading the data
# ------------------------------------------------------------
#Read the GRN data and slide features
GRN_wd = '/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/grn_CO/female_type2'
oracle_object_name = 'female_type2_fitted.celloracle.oracle'
GRN_links_after_fit, GRN_network_scores, GRN_TFs = fetch_GRN_data(oracle_object_name, GRN_wd)
slide_features = read_slide_data(feature_folder, experiment)
cluster_fusions = []
for ord_clus in order_fr_clust:
    cluster_fusions += list(itertools.combinations(clusters_of_interest, ord_clus))

In [ ]:
create_df_pickle = True
for cluster_fusion in cluster_fusions:
    combined_links, threshold = create_combined_links_for_cluster_fusion(out_path, cluster_fusion, GRN_links_after_fit, quantile=0.70)
    grn, edges_df = filter_combined_links_and_build_grn(combined_links, threshold)

    fig = edges_df.groupby(['strength', 'key']).size().unstack().plot(kind='bar', stacked=True)
    plt.xlabel('Strength')
    plt.ylabel('Count')
    plt.title('Distribution of Key and Strength')
    plt.savefig(f"{out_path}/figures/combined_links_key_strength_{cluster_fusion}{experiment}.pdf")
    plt.close()
    # ------------------------------------------------------------
    # Creating the TF lists
    # ------------------------------------------------------------
    # Step1: Filter SLIDE data wrt GRN, since SLIDE and CO gene sets are not same
    slide_features = slide_features.intersection(set(grn.nodes)) 
    slide_features_neighbors = []
    for gene in slide_features:
        slide_features_neighbors += list(grn.predecessors(gene))
    slide_tot_TF = (slide_features.union(set(slide_features_neighbors))).intersection(GRN_TFs)

    # # Step2: Creating the list of network and random TFs only for analysis for order of TF combinations >1
    # # Read network scores
    # combined_network_scores = filter_network_score_data(cluster_fusion, GRN_network_scores)
    # combined_network_scores = combined_network_scores[combined_network_scores.index.isin(list(grn.nodes))] # Since SLIDE and CO gene sets are not same
    # combined_network_scores = combined_network_scores[combined_network_scores.index.isin(GRN_TFs)] # Since I want to pick size matched set of TFs only
    # #### Choosing size matched set of TFs from network and random
    # net_match_TF = set(combined_network_scores.index[:len(slide_tot_TF)])
    # # net_rnd_TF = set(random.sample(set(GRN_TFs).intersection(set(grn.nodes)), len(slide_tot_TF)))
    # net_rnd_TF = set(random.sample(set(GRN_TFs), len(slide_tot_TF)))

    # # write the TFs to file as separate csvs
    # pd.DataFrame(slide_tot_TF).to_csv(f"{out_path}/out_files/SLIDE_LF_{cluster_fusion}_{experiment}.csv", index=False)
    # pd.DataFrame(net_match_TF).to_csv(f"{out_path}/out_files/Net_match_{cluster_fusion}_{experiment}.csv", index=False)
    # pd.DataFrame(net_rnd_TF).to_csv(f"{out_path}/out_files/Net_rnd_{cluster_fusion}_{experiment}.csv", index=False)
    # # Create a Venn diagram for the three sets
    # from matplotlib_venn import venn3
    # venn = venn3([slide_tot_TF, set(net_match_TF), set(net_rnd_TF)], ('Slide', 'Net', 'Random'))
    # plt.show()


    # ------------------------------------------------------------
    # Finally doing enrichments
    # ------------------------------------------------------------
    for ord_tf in order_fr_tfcomb:
        cc_dict = {} # Initialize the dictionary to store enrichment results for each cluster fusion and order of TF combinations
        # Step1: Deciding TF combinations for which the enrichment needs to be done
        if ord_tf == 1:
            cases = [(slide_tot_TF, 'slide')]
        elif ord_tf == 2:
            # common_TFs = slide_tot_TF.intersection(net_match_TF.intersection(net_rnd_TF)) 
            # slide_tot_TF = slide_tot_TF- common_TFs
            # net_match_TF = net_match_TF - common_TFs
            # net_rnd_TF = net_rnd_TF - common_TFs
            cases = [(slide_tot_TF, 'slide')] #[(slide_tot_TF, 'slide'),(net_match_TF, 'net'),(net_rnd_TF, 'rnd')]
        # Step2: Performing enrichment analysis
        for TFs, case in cases:
            print(f"Running enrichment for {experiment}, {cluster_fusion}, {ord_tf} TFs, case = {case}")
            cc_dict = get_SLIDE_GRN_enrichment(edges_df,cc_dict,cluster_fusion,ord_tf,slide_features,slide_starting_genes,TFs,case)
        # ------------------------------------------------------------
        # Optionally generating the dataframe to be saved
        # ------------------------------------------------------------
        if create_df_pickle == True:
            suffix = f"_{experiment}"
            print(f"Creating enrichment dataframe for {experiment}, {cluster_fusion}, {ord_tf} TFs, all cases")
            enrichment_df= create_enrichment_df(out_path, cc_dict, cluster_fusion, ord_tf, filter = None, suffix=suffix)
            print(enrichment_df['case'].value_counts())
            # dumping the dictionary to pickle file
            with open(f"{out_path}/out_files/SLIDE_LF_enrichment/cc_dict_{ord_tf}_TFs_{cluster_fusion}{suffix}.pickle", 'wb') as f:
                pickle.dump(cc_dict, f)

# Plotting enrichment scores per cluster

In [ ]:
import pickle
cluster_fusion = ('KO','WT')
experiment = 'female_type2'
ord_tf = 1
suffix = f"_{experiment}"
with open(f"{out_path}/out_files/SLIDE_LF_enrichment/cc_dict_{ord_tf}_TFs_{cluster_fusion}{suffix}.pickle", "rb") as f:
    cc_dict = pickle.load(f)

In [ ]:
# load the slide lf enriched dataframe
enrichment_df = pd.read_csv(f"{out_path}/out_files/SLIDE_LF_enrichment/enriched_df_1_TFs_('KO', 'WT')_female_type2.csv")

In [ ]:
import ast
#filter the TF and downstream LF genes into a tuple (rows should be TFs) from this df into another df
filtered_df = enrichment_df[enrichment_df['case']=='slide']
filtered_df = filtered_df[['TF', 'common']]
filtered_df['TF'] = filtered_df['TF'].apply(ast.literal_eval).apply(tuple)
filtered_df['common'] = filtered_df['common'].apply(ast.literal_eval).apply(list)
display(filtered_df)


In [ ]:
concatenated_df = pd.DataFrame()
for i in range(len(cc_dict[cluster_fusion][ord_tf])):
    used_weight_df = cc_dict[cluster_fusion][ord_tf][i][5][1]
    if used_weight_df.empty:
        continue
    used_weight_df['case'] = cc_dict[cluster_fusion][ord_tf][i][4]
    concatenated_df = pd.concat([concatenated_df, used_weight_df], ignore_index=True)
concatenated_df.to_csv(f"{out_path}/out_files/SLIDE_LF_enrichment/{ord_tf}_TFs_{cluster_fusion}{suffix}_used_weights.csv", index=False)

In [ ]:
import ast
slide_lf_enriched = pd.read_csv(f"{out_path}/out_files/SLIDE_LF_enrichment/enriched_df_{ord_tf}_TFs_{cluster_fusion}{suffix}.csv")
used_weight_df = pd.read_csv(f"{out_path}/out_files/SLIDE_LF_enrichment/{ord_tf}_TFs_{cluster_fusion}{suffix}_used_weights.csv")
slide_lf_enriched = slide_lf_enriched[slide_lf_enriched['case']=='slide']
used_weight_df = used_weight_df[used_weight_df['case']=='slide']

slide_lf_enriched = slide_lf_enriched[['TF', 'common']]
slide_lf_enriched['TF'] = slide_lf_enriched['TF'].apply(ast.literal_eval).apply(tuple)
slide_lf_enriched['common'] = slide_lf_enriched['common'].apply(ast.literal_eval).apply(list)
slide_lf_enriched = slide_lf_enriched.explode('common')
slide_lf_enriched = slide_lf_enriched.explode('TF')
slide_lf_enriched.columns = ['source', 'target']
used_weight_df = used_weight_df[used_weight_df['case']=='slide']
used_weight_df = used_weight_df[used_weight_df['strength']==1]
slide_lf_enriched = slide_lf_enriched.merge(used_weight_df, on=['source', 'target'], how='left')
slide_lf_enriched.to_csv(f"{out_path}/out_files/SLIDE_LF_enrichment/enriched_df_{ord_tf}_TFs_{cluster_fusion}{suffix}_used_weights_strong.csv", index=False)

In [ ]:
fig = slide_lf_enriched.groupby(['strength', 'key']).size().unstack().plot(kind='bar', stacked=True)
# fig = slide_lf_enriched.groupby(['strength', 'key']).plot(kind='bar', stacked=True)
plt.xlabel('Strength')
plt.ylabel('Count')
plt.title('Distribution of Key and Strength')
plt.savefig(f"{out_path}/figures/enriched_key_strength_{ord_tf}_TFs_{cluster_fusion}{suffix}_used_weights_strong.pdf")
plt.close()

In [ ]:
def read_slide_data_2(feature_folder, experiment):
    feature_files = glob.glob(f"{feature_folder}/*feature_list*")
    if experiment == 'male_type2':
        feature_data = [pd.read_csv(file, sep='\t', header=0) for file in feature_files if 'Z15' in file or 'Z29' in file]
    elif experiment == 'female_type2':
        feature_data = [pd.read_csv(file, sep='\t', header = 0) for file in feature_files if 'Z15' in file or 'Z38' in file or 'Z42' in file or 'Z55' in file]
    feature_data_df = pd.concat(feature_data)
    positive_corr_names = set(feature_data_df[feature_data_df['corrs'] >= 0]['names'])
    negative_corr_names = set(feature_data_df[feature_data_df['corrs'] < 0]['names'])
    return positive_corr_names, negative_corr_names


positive_corr_names, negative_corr_names = read_slide_data_2(feature_folder, experiment)

In [ ]:
path_to_adata = '/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/objects/female_adata_stream_input.h5ad'
adata = sc.read_h5ad(path_to_adata)
#sc.pl.umap(adata=adata, color=['C_scANVI','condition','Pdk4', 'Ucp3', 'Kcnip1', 'Sntb1', 'Abra'], frameon=False,)
sc.pl.umap(adata=adata, color=['C_scANVI','condition','Nr4a3', 'Irs2', 'Pdk4', 'Foxo3', 'Foxa3'], frameon=False,)

In [ ]:
import plotly.express as px
import pandas as pd

# Compute color proportions per TF
slide_lf_enriched['color'] = slide_lf_enriched['target'].apply(lambda gene: 'red' if gene in positive_corr_names else ('blue' if gene in negative_corr_names else 'gray'))
color_histcounts = slide_lf_enriched.groupby('source')['color'].value_counts(normalize=True).unstack(fill_value=0)

# Initialize score map
tf_name_score = {key: None for key in color_histcounts.index.values}

# Fill score map
for entry in cc_dict[cluster_fusion][ord_tf]:
    tf, cond, score_val = entry[0][0], entry[1], entry[2][0]
    if tf in tf_name_score and cond == (1,):
        tf_name_score[tf] = score_val

# Filter and sort by score
sorted_data = sorted(
    [(v, k) for k, v in tf_name_score.items() if v is not None],
    reverse=True,
    key=lambda x: x[0]
)
if not sorted_data:
    raise ValueError("No valid TF scores found for plotting.")

score, tf_name = zip(*sorted_data)
score_series = pd.Series(score, index=tf_name, name='score')

# Subset color proportions in sorted TF order
color_proportions = color_histcounts.loc[list(tf_name)]

# Save original proportions for annotation
original_props = color_proportions.copy()

# Scale each color proportion by the score (so bar height = score)
scaled = color_proportions.mul(score_series, axis=0).reset_index()
scaled = scaled.rename(columns={'index': 'source'})

# Melt both DataFrames
melted_scaled = scaled.melt(id_vars='source', var_name='color', value_name='height')
melted_props = original_props.reset_index().melt(id_vars='source', var_name='color', value_name='proportion')

# Merge both for text labels
plot_df = melted_scaled.merge(melted_props, on=['source', 'color'])

# Format the text as percentages
plot_df['text'] = (plot_df['proportion'] * 100).round(1).astype(str) + '%'

# Plot
fig = px.bar(
    plot_df,
    x='source',
    y='height',
    color='color',
    text='text',  # <- Add text here
    labels={'height': 'Score-scaled Proportion', 'source': 'TF'},
    title=f'TF Enrichment Scores with color Proportions for {experiment} - {cluster_fusion} - {ord_tf} Order'
)

# Improve text visibility inside bars
fig.update_traces(textposition='inside', insidetextanchor='middle')

# Layout tweaks
fig.update_layout(
    xaxis_title='Transcription Factor (TF)',
    yaxis_title='Enrichment Score',
    xaxis_tickangle=-90,
    font=dict(family="Arial", size=8, color="black"),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="white",
    xaxis=dict(showgrid=False, showline=True, linecolor='black', ticks='outside'),
    yaxis=dict(showgrid=False, showline=True, linecolor='black', ticks='outside')
)

fig.show()
fig.write_image(f"{out_path}/figures/0_7_TF_Enrichment_Scores_with_color_Proportions_{ord_tf}_TFs_{cluster_fusion}_{experiment}.svg", format='svg')

## Viz the TF centric GRN

In [ ]:
import ast
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go


def build_tf_edge_df(
    enrichment_df: pd.DataFrame,
    used_weight_df: pd.DataFrame,
    tf: str,
    condition: str = "KO",           # "KO" | "WT" | "both"
    strength: int | None = 1,
    gene_set: str = "all",           # "common" | "all"
    fallback: bool = True,           # when condition="KO"/"WT", borrow other cond if missing
) -> pd.DataFrame:
    """
    Returns an edge dataframe for one TF.

    condition="both" → returns ALL rows for both KO and WT (one row per target×cluster).
    condition="KO"/"WT" → primary condition's weight per target; falls back to the
                           other condition (flagged via used_condition) when fallback=True.

    Output columns: source, target, weight, cluster, strength, used_condition
    """
    gene_set = gene_set.lower()
    col = "common" if gene_set == "common" else "dwnstrm"

    enr = enrichment_df.copy()
    if enr["TF"].dtype == object:
        enr["TF"] = enr["TF"].apply(lambda v: ast.literal_eval(v) if isinstance(v, str) else v)
        enr["TF"] = enr["TF"].apply(lambda v: v[0] if isinstance(v, (tuple, list)) and len(v) == 1 else v)

    row = enr[enr["TF"].str.lower() == tf.lower()]
    if row.empty:
        raise ValueError(f"TF '{tf}' not found in enrichment_df.")

    raw = row.iloc[0][col]
    if isinstance(raw, str):
        raw = ast.literal_eval(raw)
    gene_list = list(raw)

    w = used_weight_df.copy()
    w = w[w["source"].str.lower() == tf.lower()]
    w = w[w["target"].isin(gene_list)]
    if strength is not None:
        w = w[w["strength"] == strength]

    if condition == "both":
        # Keep best |weight| per target × cluster
        w = (w.assign(_abs=w["weight"].abs())
               .sort_values("_abs", ascending=False)
               .drop_duplicates(subset=["source", "target", "cluster"])
               .drop(columns="_abs")
               .reset_index(drop=True))
        w["used_condition"] = w["cluster"]
        return w

    # Single condition with optional fallback
    other = "WT" if condition == "KO" else "KO"
    records = []
    for gene in gene_list:
        g = w[w["target"] == gene]
        primary = g[g["cluster"] == condition]
        if not primary.empty:
            best = primary.loc[primary["weight"].abs().idxmax()].to_dict()
            best["used_condition"] = condition
            records.append(best)
        elif fallback:
            secondary = g[g["cluster"] == other]
            if not secondary.empty:
                best = secondary.loc[secondary["weight"].abs().idxmax()].to_dict()
                best["used_condition"] = other
                records.append(best)

    if not records:
        raise ValueError(f"No edges resolved for TF='{tf}', condition='{condition}'.")
    return pd.DataFrame(records).reset_index(drop=True)


def plot_tf_network(
    enrichment_df: pd.DataFrame,
    used_weight_df: pd.DataFrame,
    tf: str,
    condition: str = "KO",           # "KO" | "WT" | "both"
    strength: int | None = 1,
    gene_set: str = "all",           # "common" | "all"
    highlight_common: bool = True,
    pos_color: str = "#E24B4A",      # activation  → red
    neg_color: str = "#378ADD",      # repression  → blue
    tf_node_color: str = "#7F77DD",  # TF hub      → purple
    common_border: str = "#FAC775",  # common gene → amber ring
    min_width: float = 1.5,
    max_width: float = 14.0,
    node_size: int = 20,
    height: int = 750,
    title: str | None = None,
    save_path: str | None = None,
    show: bool = False,
) -> go.Figure:
    """
    condition="KO"/"WT" : solid edges from that condition; dotted fallback from the other.
    condition="both"    : KO edges solid+bold, WT edges dotted.
    """
    edges = build_tf_edge_df(enrichment_df, used_weight_df, tf, condition, strength, gene_set)

    # Common genes for border highlight
    common_genes: set = set()
    if highlight_common and gene_set == "all":
        enr = enrichment_df.copy()
        if enr["TF"].dtype == object:
            enr["TF"] = enr["TF"].apply(lambda v: ast.literal_eval(v) if isinstance(v, str) else v)
            enr["TF"] = enr["TF"].apply(lambda v: v[0] if isinstance(v, (tuple, list)) and len(v) == 1 else v)
        r = enr[enr["TF"].str.lower() == tf.lower()]
        if not r.empty:
            raw = r.iloc[0]["common"]
            common_genes = set(ast.literal_eval(raw) if isinstance(raw, str) else raw)

    # ── Layout ───────────────────────────────────────────────────────────────
    all_targets = edges["target"].unique().tolist()
    n = len(all_targets)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
    pos = {tf: np.array([0.0, 0.0])}
    for gene, angle in zip(all_targets, angles):
        pos[gene] = np.array([np.cos(angle), np.sin(angle)])

    # ── Edge width normalisation (global across all edges) ───────────────────
    w_abs = edges["weight"].abs()
    w_min, w_max = w_abs.min(), w_abs.max()
    span = w_max - w_min if w_max > w_min else 1.0

    # ── Edge traces ──────────────────────────────────────────────────────────
    edge_traces, annotations = [], []
    for _, r in edges.iterrows():
        src, tgt, w = r["source"], r["target"], r["weight"]
        used = r["used_condition"]
        x0, y0 = pos[src]
        x1, y1 = pos[tgt]
        norm = (abs(w) - w_min) / span
        width = min_width + norm * (max_width - min_width)
        color = pos_color if w >= 0 else neg_color

        # Solid = KO (or primary); dotted = WT (or fallback)
        is_ko = (used == "KO")
        dash  = "solid" if is_ko else "dot"
        lw    = width if is_ko else max(width * 0.65, min_width)

        edge_traces.append(go.Scatter(
            x=[x0, x1, None], y=[y0, y1, None],
            mode="lines",
            line=dict(width=lw, color=color, dash=dash),
            hovertemplate=(
                f"<b>{src} → {tgt}</b><br>weight: {w:.4f}<br>"
                f"{'activation' if w >= 0 else 'repression'}<br>"
                f"condition: {used}"
                + (" <i>(fallback)</i>" if used != condition and condition != "both" else "")
                + "<extra></extra>"
            ),
            showlegend=False,
        ))
        frac = 0.78
        annotations.append(dict(
            x=x0 + frac*(x1-x0), y=y0 + frac*(y1-y0),
            ax=x0 + (frac-0.12)*(x1-x0), ay=y0 + (frac-0.12)*(y1-y0),
            xref="x", yref="y", axref="x", ayref="y",
            showarrow=True, arrowhead=2, arrowsize=1.2, arrowwidth=1.5,
            arrowcolor=color,
        ))

    # ── Node traces ──────────────────────────────────────────────────────────
    # Use highest-|weight| row per target to colour the node
    node_w = (edges.assign(_abs=edges["weight"].abs())
                   .sort_values("_abs", ascending=False)
                   .drop_duplicates(subset=["target"])
                   .set_index("target")["weight"])

    tx, ty, tlabels, tcolors, tborder_c, tborder_w, thovers = [], [], [], [], [], [], []
    for gene in all_targets:
        w = node_w[gene]
        is_common = gene in common_genes
        fill   = pos_color if w >= 0 else neg_color
        bcolor = common_border if (highlight_common and is_common) else "white"
        bwidth = 3.0 if (highlight_common and is_common) else 1.5
        tx.append(pos[gene][0]); ty.append(pos[gene][1])
        tlabels.append(f"{'★ ' if is_common else ''}{gene}")
        tcolors.append(fill); tborder_c.append(bcolor); tborder_w.append(bwidth)
        thovers.append(
            f"<b>{gene}</b><br>weight: {w:.4f}<br>"
            f"{'activation' if w >= 0 else 'repression'}<br>"
            f"{'<i>common / LF-enriched</i>' if is_common else '<i>downstream only</i>'}"
            "<extra></extra>"
        )

    target_trace = go.Scatter(
        x=tx, y=ty, mode="markers+text",
        marker=dict(size=node_size, color=tcolors,
                    line=dict(width=tborder_w, color=tborder_c)),
        text=tlabels, textposition="top center",
        textfont=dict(size=10, family="Arial"),
        hovertemplate=thovers, showlegend=False,
    )
    tf_trace = go.Scatter(
        x=[0], y=[0], mode="markers+text",
        marker=dict(size=node_size*2, color=tf_node_color, line=dict(width=2, color="white")),
        text=[tf], textposition="middle center",
        textfont=dict(size=13, color="white", family="Arial"),
        hovertemplate=f"<b>{tf}</b> (TF hub)<extra></extra>",
        showlegend=False,
    )

    # ── Legend ────────────────────────────────────────────────────────────────
    legend = [
        go.Scatter(x=[None], y=[None], mode="lines",
                   line=dict(width=4, color=pos_color), name="Activation (w > 0)"),
        go.Scatter(x=[None], y=[None], mode="lines",
                   line=dict(width=4, color=neg_color), name="Repression (w < 0)"),
        go.Scatter(x=[None], y=[None], mode="markers",
                   marker=dict(size=10, color=tf_node_color), name=f"{tf} (TF)"),
    ]
    if condition == "both":
        legend += [
            go.Scatter(x=[None], y=[None], mode="lines",
                       line=dict(width=3, color="gray", dash="solid"), name="KO (solid)"),
            go.Scatter(x=[None], y=[None], mode="lines",
                       line=dict(width=2, color="gray", dash="dot"),   name="WT (dotted)"),
        ]
    elif edges["used_condition"].ne(condition).any():
        other = "WT" if condition == "KO" else "KO"
        legend.append(go.Scatter(x=[None], y=[None], mode="lines",
                                 line=dict(width=2, color="gray", dash="dot"),
                                 name=f"fallback weight ({other})"))
    if highlight_common and gene_set == "all":
        legend += [
            go.Scatter(x=[None], y=[None], mode="markers",
                       marker=dict(size=10, color="gray", line=dict(width=3, color=common_border)),
                       name="★ common / LF-enriched"),
            go.Scatter(x=[None], y=[None], mode="markers",
                       marker=dict(size=10, color="gray", line=dict(width=1.5, color="white")),
                       name="downstream only"),
        ]

    n_common = sum(g in common_genes for g in all_targets)
    auto_title = (
        f"<b>{tf}</b> — {condition} | {gene_set} genes "
        f"({len(all_targets)} targets"
        + (f", {n_common} common" if gene_set == "all" else "")
        + ")"
    )

    fig = go.Figure(
        data=edge_traces + [target_trace, tf_trace] + legend,
        layout=go.Layout(
            title=dict(text=title or auto_title, font=dict(size=14, family="Arial"), x=0.5),
            height=height, showlegend=True,
            legend=dict(x=1.01, y=0.95, bgcolor="rgba(255,255,255,0.85)",
                        bordercolor="lightgray", borderwidth=1, font=dict(size=11)),
            hovermode="closest",
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor="y"),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            plot_bgcolor="white", paper_bgcolor="white",
            margin=dict(l=20, r=180, t=60, b=20),
            annotations=annotations,
        ),
    )
    if show:
        fig.show()
    if save_path:
        fig.write_image(save_path, format=save_path.rsplit(".", 1)[-1])
        print(f"Saved → {save_path}")
    return fig




In [ ]:
# ── Usage ─────────────────────────────────────────────────────────────────────
fig = plot_tf_network(enrichment_df, used_weight_df, tf="Foxa3", condition="KO",
                      gene_set="all", highlight_common=True)
fig.show()

In [ ]:

# WT only
fig = plot_tf_network(enrichment_df, used_weight_df, tf="Foxa3", condition="WT",
                      gene_set="all", highlight_common=True)
fig.show()
